# 01 · Dim_Geografía
**IDY1102 · EP3 · DataRetail 360**

| Campo | Detalle |
|---|---|
| Responsable | [Nombre del integrante] |
| Tabla destino | `dim_geografia` |
| SCD | **Tipo 2**: provincia, región, zona_comercial · Tipo 1: comuna, país, coordenadas |
| Clave natural | `codigo_comuna` (código CUT) |
| Depende de | — (se ejecuta primero) |
| Fuentes | `geografia_snapshot_1.csv` y `geografia_snapshot_2.csv` |
| Alimenta a | `dim_sucursal` y `dim_cliente` |

> **Cómo usar esta plantilla.** Las celdas marcadas **✏️ TODO** son la parte que debe implementar el integrante
> responsable (las reglas de calidad). Todo lo demás ya está conectado a los módulos compartidos de `src/`
> (carga idempotente, SCD, auditoría, logging). Antes de commitear: *Restart & Run All* debe terminar sin errores.

## 0. Setup

In [1]:
import os, sys
from datetime import date
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

from src import db, extract, medallion, quality as q, schema
from src.config import get_settings
from src.logger import get_logger

PROCESO = "dim_geografia"
log = get_logger(PROCESO)
cfg = get_settings()
engine = db.get_engine(cfg)
schema.crear_esquema(engine)                  # idempotente: crea solo lo que falta
schema.sembrar_miembros_desconocidos(engine)  # miembro -1 "No informado" en cada dimensión
pd.set_option("display.width", 160)

2026-10-25 19:09:22,812 | INFO     | datarretail360.db | Conectado a sqlserver (DataRetail360_DW)


## 1. Parámetros

In [2]:
# Snapshot a procesar: lo define run_pipeline.py (variable de entorno SNAPSHOT) o puedes cambiarlo a mano.
SNAPSHOT = int(os.getenv("SNAPSHOT", "1"))
FECHAS_EFECTIVAS = {1: date(2025, 1, 1), 2: date(2025, 6, 1)}   # desde cuándo rige cada snapshot
FECHA_EFECTIVA = FECHAS_EFECTIVAS[SNAPSHOT]
ARCHIVO = cfg.data_dir / f"geografia_snapshot_{SNAPSHOT}.csv"
print(f"Snapshot {SNAPSHOT} · fecha efectiva {FECHA_EFECTIVA} · {ARCHIVO.name}")

Snapshot 1 · fecha efectiva 2025-01-01 · geografia_snapshot_1.csv


## 2. Extract
Lee la fuente cruda y la guarda **sin cambios** en la capa Bronze.

In [3]:
def extraer(ruta):
    """Lee la fuente cruda y la deja intacta en la capa Bronze."""
    df = extract.leer_csv(ruta)
    medallion.guardar_capa(df, "bronze", f"geografia_snapshot_{SNAPSHOT}")
    return df

## 3. Transform  ✏️ TODO
**Implementa al menos 3 reglas de calidad** (la rúbrica del EP3 y la Actividad 2.2 lo piden). Estas son las que traen los datos de ejemplo:

| Problema en los datos de ejemplo | Dimensión de calidad | Pista |
|---|---|---|
| `codigo_comuna` repetido (Santiago aparece dos veces en el snapshot 1) | Unicidad | `duplicated`, `q.separar_rechazos` |
| Textos con espacios y formatos distintos | Consistencia | `q.normalizar_texto` |
| Coordenadas imposibles (una fila trae 999) | Validez | `pd.to_numeric(errors='coerce')` + `between` (Chile: lat −56…−17, lon −110…−66) |
| Provincia y región vacías en una fila | Completitud | máscara con `isna()` / `== ''` y `q.separar_rechazos` |

In [4]:
# Contrato de salida de transformar(): columnas que espera la carga (no lo borres)
COLUMNAS_SALIDA = ["codigo_comuna", "comuna", "provincia", "region", "pais", "zona_comercial", "latitud", "longitud"]

In [ ]:
import pandas as pd
import src.quality as q

def transformar(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Limpia y valida. Retorna (df_limpio, df_rechazos).
    """
    if df is None or df.empty:
        return pd.DataFrame(), pd.DataFrame()

    df_work = df.copy()
    rechazos = []

    # 1. Normalizar codigo_comuna y textos
    if 'codigo_comuna' in df_work.columns:
        df_work['codigo_comuna'] = df_work['codigo_comuna'].astype(str).str.strip()

    columnas_texto = ['comuna', 'provincia', 'region', 'zona_comercial', 'pais']
    for col in columnas_texto:
        if col in df_work.columns:
            df_work[col] = q.normalizar_texto(df_work[col])

    # 2. Convertir latitud y longitud a numérico
    if 'lat' in df_work.columns:
        df_work['lat'] = pd.to_numeric(df_work['lat'], errors='coerce')
    if 'long' in df_work.columns:
        df_work['long'] = pd.to_numeric(df_work['long'], errors='coerce')

    # 3. Validar y separar coordenadas inválidas o fuera de rango para Chile
    if 'lat' in df_work.columns and 'long' in df_work.columns:
        mask_coord_invalida = (
            df_work['lat'].isna() | df_work['long'].isna() |
            ~df_work['lat'].between(-56.0, -17.0) |
            ~df_work['long'].between(-109.0, -66.0)
        )
        df_work, r_coord = q.separar_rechazos(df_work, mask_coord_invalida, "Coordenada invalida o fuera de rango")
        if not r_coord.empty:
            rechazos.append(r_coord)

    # 4. Separar duplicados por codigo_comuna
    if 'codigo_comuna' in df_work.columns:
        mask_duplicado = df_work.duplicated(subset=['codigo_comuna'], keep='first')
        df_work, r_dup = q.separar_rechazos(df_work, mask_duplicado, "Codigo comuna duplicado")
        if not r_dup.empty:
            rechazos.append(r_dup)


    df_limpio = df_work
    
    df_rechazos_cons = q.consolidar_rechazos(*rechazos) if rechazos else pd.DataFrame()

    return df_limpio, df_rechazos_cons

## 4. Load
Carga idempotente con el helper compartido y ejecución completa con auditoría (`Ejecucion`).

In [11]:
ATRIBUTOS_SCD2 = ["provincia", "region", "zona_comercial"]   # un cambio aquí crea una versión nueva
ATRIBUTOS_SCD1 = ["comuna", "pais", "latitud", "longitud"]   # un cambio aquí sobrescribe

def cargar(df_limpio: pd.DataFrame, df_rechazos: pd.DataFrame) -> dict:
    """Registra los rechazos y carga la dimensión aplicando SCD Tipo 2."""
    assert set(COLUMNAS_SALIDA) <= set(df_limpio.columns), "transformar() no devolvió las columnas esperadas"
    db.registrar_rechazos(engine, PROCESO, df_rechazos)
    return db.aplicar_scd2(engine, "dim_geografia", df_limpio, clave_natural="codigo_comuna",
                           atributos_scd2=ATRIBUTOS_SCD2, fecha_efectiva=FECHA_EFECTIVA,
                           key_col="geografia_key", atributos_scd1=ATRIBUTOS_SCD1)

In [14]:
with db.Ejecucion(engine, PROCESO) as run:
    df_crudo = extraer(ARCHIVO)
    run.leidas = len(df_crudo)
    df_limpio, df_rechazos = transformar(df_crudo)
    medallion.guardar_capa(df_limpio, "silver", f"geografia_snapshot_{SNAPSHOT}")
    resumen = cargar(df_limpio, df_rechazos)
    run.cargadas = resumen["nuevos"] + resumen["versiones_nuevas"]
    run.rechazadas = len(df_rechazos)
resumen

2026-10-25 19:15:34,261 | INFO     | datarretail360.db | [dim_geografia] INICIO
2026-10-25 19:15:34,264 | INFO     | datarretail360.extract | Extraídas 16 filas desde geografia_snapshot_1.csv
2026-10-25 19:15:34,268 | INFO     | datarretail360.medallion | Capa bronze: 16 filas guardadas en geografia_snapshot_1.parquet
2026-10-25 19:15:34,281 | INFO     | datarretail360.medallion | Capa silver: 15 filas guardadas en geografia_snapshot_1.parquet
2026-10-25 19:15:34,310 | INFO     | datarretail360.db | Carga auditoria_rechazos: {'insertadas': 1, 'ya_existian': 0, 'duplicadas_en_lote': 0}
2026-10-25 19:15:34,311 | WARNING  | datarretail360.db | [dim_geografia] 1 filas rechazadas (nuevas en auditoría: 1)
2026-10-25 19:15:34,359 | INFO     | datarretail360.db | SCD2 dim_geografia (efectiva 2025-01-01): {'nuevos': 15, 'versiones_nuevas': 0, 'actualizados_scd1': 0, 'sin_cambio': 0, 'omitidos': 0}
2026-10-25 19:15:34,360 | INFO     | datarretail360.db | [dim_geografia] FIN OK leidas=16 cargadas

{'nuevos': 15,
 'versiones_nuevas': 0,
 'actualizados_scd1': 0,
 'sin_cambio': 0,
 'omitidos': 0}

## 5. Validación
Chequeos automáticos sobre lo ya cargado. Si algo está mal, el notebook **falla** (no sigue en silencio).

In [15]:
def validar():
    dim = db.leer_tabla(engine, "dim_geografia")
    problemas = q.chequear_scd2(dim, "codigo_comuna", "geografia_key")
    assert not problemas, problemas
    assert (dim["geografia_key"] == -1).sum() == 1, "debe existir exactamente un miembro 'No informado' (-1)"
    log.info("Validación de dim_geografia OK (%s filas)", len(dim))

validar()

2026-10-25 19:15:42,885 | INFO     | datarretail360.dim_geografia | Validación de dim_geografia OK (16 filas)


In [16]:
# Idempotencia: volver a cargar los MISMOS datos no debe cambiar nada
filas_antes = len(db.leer_tabla(engine, "dim_geografia"))
resumen_2 = cargar(df_limpio, pd.DataFrame(columns=q.COLUMNAS_RECHAZO))
assert len(db.leer_tabla(engine, "dim_geografia")) == filas_antes, "la carga no es idempotente"
print("Idempotencia OK:", resumen_2)

2026-10-25 19:15:46,020 | INFO     | datarretail360.db | SCD2 dim_geografia (efectiva 2025-01-01): {'nuevos': 0, 'versiones_nuevas': 0, 'actualizados_scd1': 15, 'sin_cambio': 0, 'omitidos': 0}
Idempotencia OK: {'nuevos': 0, 'versiones_nuevas': 0, 'actualizados_scd1': 15, 'sin_cambio': 0, 'omitidos': 0}


## 6. Evidencia
Estas salidas son las que se capturan para el informe (apartados 6, 7 y 9 de la pauta EP3).

In [17]:
def conteos_por_capa() -> pd.DataFrame:
    """Filas en cada capa Medallion para este snapshot: trazabilidad fuente → bronze → silver → gold."""
    bronze = medallion.leer_capa("bronze", f"geografia_snapshot_{SNAPSHOT}")
    silver = medallion.leer_capa("silver", f"geografia_snapshot_{SNAPSHOT}")
    gold = db.leer_tabla(engine, "dim_geografia", solo_vigentes=True)
    gold = gold[gold["geografia_key"] != -1]
    return pd.DataFrame({"capa": ["bronze (crudo)", "silver (limpio)", "gold (vigentes en el DW)"],
                          "filas": [len(bronze), len(silver), len(gold)]})

conteos_por_capa()

,capa,filas
0,bronze (crudo),16
1,silver (limpio),15
2,gold (vigentes en el DW),15


In [20]:
# Historia SCD2 de un registro de ejemplo (Chillán, que cambia de zona comercial en el snapshot 2): una fila por versión
dim = db.leer_tabla(engine, "dim_geografia")
versiones = dim.groupby("codigo_comuna").size()
ejemplos = versiones[versiones > 1].index[:1]
dim[dim["codigo_comuna"].isin(ejemplos)].sort_values(["codigo_comuna", "valido_desde"])

,geografia_key,codigo_comuna,comuna,provincia,region,pais,zona_comercial,latitud,longitud,valido_desde,valido_hasta,es_actual


In [19]:
# Rechazos de este proceso y últimas ejecuciones registradas (auditoría)
rech = db.leer_tabla(engine, "auditoria_rechazos")
display(rech[rech["proceso"] == PROCESO].groupby("motivo_rechazo").size().rename("filas").to_frame())
ejec = db.leer_tabla(engine, "auditoria_ejecuciones")
ejec[ejec["proceso"] == PROCESO].tail(4)[["inicio", "estado", "filas_leidas", "filas_cargadas", "filas_rechazadas"]]

,filas
motivo_rechazo,
Codigo comuna duplicado,1


,inicio,estado,filas_leidas,filas_cargadas,filas_rechazadas
1,2026-10-25 19:11:00.607,ERROR,16,0,0
2,2026-10-25 19:12:42.537,ERROR,16,0,0
3,2026-10-25 19:13:50.310,ERROR,16,0,0
4,2026-10-25 19:15:34.260,OK,16,15,1


## 7. Decisiones de diseño (completar)
- ¿Por qué `zona_comercial` es SCD2 y `comuna` es SCD1? Justifica con costo-beneficio (costo de filas extra vs. pérdida de historia).
- ¿Qué pasa con ventas históricas de una comuna cuando cambia su zona? Explícalo con la evidencia de la sección 6.
- Jerarquía de la dimensión: comuna → provincia → región → país.